This notebook extends the ["Parallel Execution (Part 2)"](./Parallel%20Execution%20(Part%202).ipynb) example.

In [ ]:
!pip install -q langgraph

In [ ]:
import operator
import time

from IPython.display import HTML, Image
from google.colab import userdata
from langchain_core.runnables import Runnable, RunnableConfig
from langgraph.checkpoint.base import BaseCheckpointSaver
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import Command, Interrupt, interrupt
from pathlib import Path
from typing import Annotated, TypedDict, List

def print_interrupts(interrupts: List[Interrupt]):
    for el in interrupts:
        display(HTML(f'<div style="border: 1px dashed red; margin: 5px; padding: 10px; white-space: pre-wrap;">{el.value}</div>'))

def display_graph(runnable: Runnable, output_png: Path) -> None:
    with output_png.open(mode="wb") as file:
        file.write(runnable.get_graph().draw_mermaid_png())

    display(Image(output_png, format="png"))

def explore_checkpoints(checkpoint_saver: BaseCheckpointSaver, config: RunnableConfig):
    checkpoints = list(checkpoint_saver.list(config))
    print(f"There are {len(checkpoints)} checkpoints in total:")
    for checkpoint in reversed(checkpoints):
        print(checkpoint)

def explore_state_history(compiled_state_graph: CompiledStateGraph, config: RunnableConfig):
    state_history = list(compiled_state_graph.get_state_history(config))

    for snapshot in reversed(state_history):
        print(f"Step: {snapshot.metadata['step']}")
        print("Current state:")
        print(snapshot.values)
        print(f"Next: {snapshot.next}")
        print()

In [ ]:
checkpointer = InMemorySaver()

def merge_findings(a: dict[str, str], b: dict[str, str]) -> dict[str, str]:
    # `b` overwrites `a` in case of conflicts.
    # This fragment can be modified to raise an error instead.
    return {**a, **b}

class ResearchState(TypedDict):
    topic: str
    findings: Annotated[dict[str, str], merge_findings]
    verification_requests: Annotated[list[str], operator.add]
    verified: Annotated[set[str], operator.or_]
    blocked: Annotated[set[str], operator.or_]
    summary: str

In [ ]:
def wikipedia(state: ResearchState):
    print("[WIKIPEDIA] node is executing")
    time.sleep(2)

    return { "findings": { "wikipedia": f"[WIKIPEDIA] {state['topic']}" }, "verification_requests": ["wikipedia"] }

def news(state: ResearchState):
    print("[LIVE NEWS] node is executing")
    time.sleep(2)

    return { "findings": { "news": f"[LIVE NEWS] Sensational! {state['topic']}" }, "verification_requests": ["news"] }

def arxiv(state: ResearchState):
    print("[ARXIV] node is executing")
    time.sleep(2)

    return { "findings": { "arxiv": f"[ARXIV] The theoretical hyperchaotic entanglement of relativities related to \"{state['topic']}\"" } }

def verify(state: ResearchState):
    current_verified = state.get('verified', set())
    current_blocked = state.get('blocked', set())

    new_verified = set()
    new_blocked = set()

    for request in state.get('verification_requests', {}):
        if request in current_verified or request in current_blocked:
            continue

        decision = interrupt(f"Pending verification:\n{state['findings'][request]}")
        if decision:
            new_verified.add(request)
        else:
            new_blocked.add(request)

    return { "verified": new_verified, "blocked": new_blocked }

def _annotate_finding(finding_key: str, state: ResearchState):
    if finding_key in state.get('verified', set()):
        return " (verified)"
    elif finding_key in state.get('blocked', set()):
        return " (blocked)"
    else:
        return ""

def merge(state: ResearchState):
    if len(state.get('verification_requests', [])) > len(state.get('verified', set())) + len(state.get('blocked', set())):
        print("There are pending verification requests. Merge node will be skipped.")
        return {}

    bullets = "\n".join(f" - {finding}{_annotate_finding(key, state)}" for key, finding in state.get("findings", {}).items())
    return { "summary": f"Summary:\n{bullets}" }

In [ ]:
graph_builder = StateGraph(ResearchState)
graph_builder.add_node("wikipedia", wikipedia)
graph_builder.add_node("news", news)
graph_builder.add_node("arxiv", arxiv)
graph_builder.add_node("verifier", verify)
graph_builder.add_node("merge", merge)

# NOTE: The three nodes ("wikipedia", "news", "arxiv") will be executed in parallel, then two of them will be verified and only after that they will be merged together.
graph_builder.add_edge(START, "wikipedia")
graph_builder.add_edge(START, "news")
graph_builder.add_edge(START, "arxiv")

graph_builder.add_edge("wikipedia", "verifier")
graph_builder.add_edge("news", "verifier")
graph_builder.add_edge("verifier", "merge")
graph_builder.add_edge("arxiv", "merge")

graph_builder.add_edge("merge", END)

# A checkpointer is necessary in order to resume the execution later.
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
display_graph(graph, Path("/content/graph.png"))

In [ ]:
thread1_config = { "configurable": { "thread_id": "thread_3" } }
thread1_result = graph.invoke(input={ "topic": "Maximum snooker break" }, config=thread1_config)

In [ ]:
print_interrupts(thread1_result['__interrupt__'])

In [ ]:
# NOTE: Look at the pending writes for the last checkpoint
explore_checkpoints(checkpointer, thread1_config)

In [ ]:
# Verify the first source
thread1_result = graph.invoke(input=Command(resume=True), config=thread1_config)

In [ ]:
print_interrupts(thread1_result['__interrupt__'])

In [ ]:
# NOTE: Look at the pending writes for the last checkpoint
explore_checkpoints(checkpointer, thread1_config)

In [ ]:
# Reject the second source
thread1_result = graph.invoke(input=Command(resume=False), config=thread1_config)

In [ ]:
print(thread1_result['summary'])

In [ ]:
explore_checkpoints(checkpointer, thread1_config)

In [ ]:
explore_state_history(graph, thread1_config)